In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StringType

# ---- Config ----
catalog    = "firstdeproject"
table_name = "sales_details"                      # change per table
full_table = f"{catalog}.bronze.{table_name}"
key_cols   = []                            # business key, e.g. ["order_id"]; empty = full-row match


# ============================================================
# 1. Read bronze table into df
# ============================================================
df = spark.table(full_table)
print(f"Rows: {df.count()} | Columns: {len(df.columns)}")
df.show(15, truncate=False)

audit_cols = [c for c in df.columns if c.startswith("_")]   # lineage columns from ingestion
data_cols  = [c for c in df.columns if c not in audit_cols]
str_cols   = [f.name for f in df.schema.fields
              if isinstance(f.dataType, StringType) and f.name not in audit_cols]



In [0]:
date_cols = {"sls_order_dt": "yyyyMMdd", "sls_ship_dt": "yyyyMMdd", "sls_due_dt": "yyyyMMdd"}  # column: source format
for c, fmt in date_cols.items():
    df = df.withColumn(c, F.try_to_date(F.when(F.trim(F.col(c).cast("string")) == "0", None)
                                         .otherwise(F.trim(F.col(c).cast("string"))), fmt))
print(df.dtypes)
df.show(5, truncate=False)

In [0]:
for c in str_cols:
        cleaned = F.trim(F.regexp_replace(F.col(c), r"\s+", " "))
        df = df.withColumn(c, F.when(cleaned == "", None).otherwise(cleaned))
print("Whitespace cleaned. Empty strings converted to NULL.")

In [0]:
# ============================================================
# 2. Find duplicates
# ============================================================

df.exceptAll(df.dropDuplicates()).show()

In [0]:
# ============================================================
# Check distinct values 
# ============================================================


for col in str_cols:
    df.select(col).distinct().show(10)


In [0]:
# ============================================================
# Convert price to numerical column with decimal, standardize colun values for modeling
# ============================================================


df = df.withColumn("sls_price", F.col("sls_price").cast("double"))
df = df.withColumn("sls_quantity", F.col("sls_quantity").cast("INT"))

df.show(truncate=False)

In [0]:

# ============================================================
# Write cleaned output to silver
# ============================================================


df.write.mode("overwrite") \
  .option("overwriteSchema", "true") \
  .saveAsTable(f"{catalog}.silver.{table_name}")

print(f"Written to {catalog}.silver.{table_name}")